In [24]:
%pip install PyPDF2
%pip install requests
%pip install -U langchain-mistralai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [25]:
import os
import fitz
import tiktoken
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    ToolMessage,
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import END, StateGraph, START
from langchain_openai import ChatOpenAI
from typing import List
from dataclasses import dataclass, field
from PyPDF2 import PdfReader


In [26]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain.document_loaders import PyPDFLoader
from langchain_mistralai import ChatMistralAI

In [28]:
@dataclass
class State:
    pdf_name: List[str] = field(default_factory=list)
    workflow_steps: int = 0
    pdf_content: str = ""
    chunks: List[str] = field(default_factory=list)
    sector: List[str] = field(default_factory=list)
    due_date: List[str] = field(default_factory=list)
    location: List[str] = field(default_factory=list)
    required_certifications: List[str] = field(default_factory=list)
    minimum_experience: List[str] = field(default_factory=list)
    similar_project_references: List[str] = field(default_factory=list)
    financial_capacity: List[str] = field(default_factory=list)
    programming_languages: List[str] = field(default_factory=list)
    information_security: List[str] = field(default_factory=list)
    software_quality: List[str] = field(default_factory=list)
    computer_networks: List[str] = field(default_factory=list)
    IT_infrastructure: List[str] = field(default_factory=list)
    third_party_systems_integration: List[str] = field(default_factory=list)
    deployment: List[str] = field(default_factory=list)
    maintainability: List[str] = field(default_factory=list)
    training: List[str] = field(default_factory=list)
    project_resources: List[str] = field(default_factory=list)
    confidentiality: List[str] = field(default_factory=list)
    project_management_approaches: List[str] = field(default_factory=list)
    project_management_tools: List[str] = field(default_factory=list)
    development_methods: List[str] = field(default_factory=list)
    technical_skills: List[str] = field(default_factory=list)
    legal_compliance: List[str] = field(default_factory=list)
    technical_support_and_maintenance: List[str] = field(default_factory=list)
    ability_to_meet_deadlines: List[str] = field(default_factory=list)
    solution_scalability: List[str] = field(default_factory=list)


In [29]:
def function_init(state: State) -> State:
    state.workflow_steps = 1
    state.pdf_name = ["1.pdf"]
    state.pdf_content = ""
    state.chunks = []
    state.sector = []
    state.due_date = []
    state.location = []
    state.required_certifications = []
    state.minimum_experience = []
    state.similar_project_references = []
    state.financial_capacity = []
    state.programming_languages = []
    state.information_security = []
    state.software_quality = []
    state.computer_networks = []
    state.IT_infrastructure = []
    state.third_party_systems_integration = []
    state.deployment = []
    state.maintainability = []
    state.training = []
    state.project_resources = []
    state.confidentiality = []
    state.project_management_approaches = []
    state.project_management_tools = []
    state.development_methods = []
    state.technical_skills = []
    state.legal_compliance = []
    state.technical_support_and_maintenance = []
    state.ability_to_meet_deadlines = []
    state.solution_scalability = []
    return state


In [30]:
def read_pdf(state: State) -> State:
    pdf_content = ""
    for file_path in state.pdf_name: 
        with open(file_path, "rb") as file:
            reader = PdfReader(file)
            num_pages = len(reader.pages)
            for page_num in range(num_pages):
                page = reader.pages[page_num]
                pdf_content += page.extract_text()
    state.pdf_content = pdf_content
    return state

In [31]:
def extract_text_from_pdf(state: State) -> State:
    """Extracts text from a PDF file."""
    state.workflow_steps += 1
    pdfs = state.pdf_name
    try:
        for file_path in pdfs:
            loader = PyPDFLoader(file_path)
            documents = loader.load()
            text_splitter = SemanticChunker(
                OpenAIEmbeddings(), breakpoint_threshold_type="standard_deviation"
            )
            semantic_chunks = text_splitter.create_documents([d.page_content for d in documents])
    except Exception as e:
        print(f"Error processing PDF file: {e}")
    state.chunks = semantic_chunks
    return state

In [32]:
chain_gpt_4o_mini = ChatOpenAI(model="gpt-4o", temperature=0.3, top_p=1)
open_mistral_7b  = ChatMistralAI(model="open-mixtral-8x22b", temperature=0.2, api_key='')

def result(state: State) -> State:
    prompt = f"""
    You are an expert in analyzing and extracting information from Request for Proposal (RFP) documents.

    **Objective:**
    1. Extract and format the provided RFP text into a detailed, structured text format for each attribute. Ensure thorough coverage of each section and capture multiple entries where applicable.
    2. Summarize the detailed text while retaining all relevant details.

    **Output Format:**
    First, present the response as a detailed and structured text summary for each attribute, followed by the references extracted from the RFP text. Use the following format for each attribute:

    **Sector:**
    - **Detailed Summary:**
      [Detailed summary of the sector and focus areas mentioned in the RFP. Include specific sectors targeted, the purpose of the RFP, and any relevant programs or initiatives.]
    - **References from RFP text:**
      [List the references extracted from the RFP text that support the detailed summary.]

    **Dates:**
    - **Detailed Summary:**
      - Release RFP date: [Date]
      - Questions due date: [Date]
      - RFP response deadline: [Date]
      - Selection and award of contract date: [Date]
      - Others: [Additional dates or milestones such as follow-up actions or decision dates.]
    - **References from RFP text:**
      [List the references extracted from the RFP text that support the detailed summary.]

    **Location:**
    - **Detailed Summary:**
      [Detailed information about the location requirements mentioned in the RFP, including geographical scope, site specifics, or any regional considerations.]
    - **References from RFP text:**
      [List the references extracted from the RFP text that support the detailed summary.]

    **Skills and References:**
    - **Minimum experience:**
      - **Detailed Summary:**
        [Detailed description of the required experience, including years, specific skills, and types of projects or roles.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Required certifications:**
      - **Detailed Summary:**
        [List of required certifications, with explanations of their relevance or importance.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Similar project references:**
      - **Detailed Summary:**
        [Detailed requirements for providing references, including the format, information needed, and any evaluation criteria.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]

    **Infrastructure:**
    - **IT infrastructure:**
      - **Detailed Summary:**
        [Detailed specifications for IT infrastructure, including hardware, software, network components, and integration needs.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Network infrastructure:**
      - **Detailed Summary:**
        [Details on network requirements, including bandwidth, connectivity, and security measures.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Virtualization:**
      - **Detailed Summary:**
        [Information on virtualization needs, if applicable, including types of virtualization technologies and their purposes.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]

    **Technical Skills:**
    - **Programming languages:**
      - **Detailed Summary:**
        [Detailed list of programming languages required, including versions and usage contexts.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Cloud computing, data management, AI skills:**
      - **Detailed Summary:**
        [Skills and technologies required for cloud computing, data management, and AI, including specific tools or platforms.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Cybersecurity, DevOps, Big Data skills:**
      - **Detailed Summary:**
        [Requirements for cybersecurity, DevOps practices, and big data management, including relevant tools and methodologies.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **IoT, network, telecommunications, blockchain skills:**
      - **Detailed Summary:**
        [Skills related to IoT, networking, telecommunications, and blockchain, including their relevance to the project.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Automation, orchestration, data analysis skills:**
      - **Detailed Summary:**
        [Skills required for automation, orchestration, and data analysis, including specific technologies and methodologies.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Others:**
      - **Detailed Summary:**
        [Any additional technical skills required.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]

    **Requested Solution Quality:**
    - **Maintainability:**
      - **Detailed Summary:**
        [Detailed requirements for maintainability, including ease of updates, ongoing management, and support.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Reliability:**
      - **Detailed Summary:**
        [Criteria for reliability, including uptime requirements, fault tolerance, and redundancy.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Flexibility:**
      - **Detailed Summary:**
        [Requirements for flexibility, including customization options and adaptability to changing needs.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Integrity:**
      - **Detailed Summary:**
        [Details on integrity requirements, including data protection, security measures, and compliance standards.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Availability:**
      - **Detailed Summary:**
        [Requirements for availability, including uptime guarantees, support, and disaster recovery plans.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Solution scalability:**
      - **Detailed Summary:**
        [Scalability requirements, including performance under load, ability to handle growth, and scaling strategies.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Others:**
      - **Detailed Summary:**
        [Any additional quality requirements.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]

    **Project Management and Resources:**
    - **Project management approaches:**
      - **Detailed Summary:**
        [Detailed description of required project management approaches, including methodologies, standards, and practices.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Project management tools:**
      - **Detailed Summary:**
        [Tools required for project management, including software, systems, and their functionalities.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Development methods:**
      - **Detailed Summary:**
        [Development methods to be used, including Agile, Waterfall, or others, with explanations of their application.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Project resources:**
      - **Detailed Summary:**
        [Details on required project resources, including team composition, roles, and responsibilities.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Training:**
      - **Detailed Summary:**
        [Training requirements, including types of training, delivery methods, and topics covered.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]
    - **Deployment:**
      - **Detailed Summary:**
        [Deployment requirements, including processes, methodologies, and any special considerations.]
      - **References from RFP text:**
        [List the references extracted from the RFP text that support the detailed summary.]

    **Technical Support and Maintenance:**
    - **Detailed Summary:**
      [Comprehensive details on technical support and maintenance requirements, including support levels, response times, and maintenance agreements.]
    - **References from RFP text:**
      [List the references extracted from the RFP text that support the detailed summary.]

    **Legal Compliance:**
    - **Detailed Summary:**
      [Detailed requirements for legal compliance, including relevant laws, regulations, and industry standards that must be adhered to.]
    - **References from RFP text:**
      [List the references extracted from the RFP text that support the detailed summary.]

    **Instructions:**
    1. Carefully read the provided RFP text.
    2. Extract and detail the references that address each attribute.
    3. Provide a thorough explanation for each attribute using the format outlined above.
    4. Ensure that each section is covered comprehensively and that all significant information is included.
    5. Note any discussions related to attributes that are not directly addressed but are relevant.
    6. You can read the text and extract attribute by attribute, which means that for each attribute you reread the text.
    7. Treat each enumerated question or list in the RFP text as a source of important details to be captured under the relevant JSON attributes, ensuring no significant information is omitted.

    Here is the provided RFP text for extraction:
    {state.pdf_content}

    **Summary:**
    After providing the detailed extraction, provide a summarized version of the text, highlighting the most important points from each section.
    """

    response = chain_gpt_4o_mini.invoke([HumanMessage(content=prompt)])
    answer = response.content.strip()
    print("Extracted Information:", answer)

    return state


c:\Users\ayoub\OneDrive\Desktop\5idma\rfps\.venv\Lib\site-packages\langchain_core\utils\utils.py:161: UserWarning: WARNING! top_p is not default parameter.
                top_p was transferred to model_kwargs.
                Please confirm that top_p is what you intended.
  warnings.warn(


In [33]:
# Define a LangChain graph
workflow = StateGraph(State)

# Initial function
workflow.add_node("init", function_init)
workflow.add_edge('init', 'read_pdf')

# Extract text from PDF
workflow.add_node("read_pdf", read_pdf)
workflow.add_edge('read_pdf', 'result')

# Filter by sector
workflow.add_node("result", result)


# Set entry and finish points
workflow.set_entry_point("init")
workflow.set_finish_point("result")

# Compile the workflow
app = workflow.compile()


In [34]:
# Test function for LangChain workflow
initstate = State()

for output in app.stream(initstate):
    for key, value in output.items():
        print(f"========== {key} output: ========")
    print(">>>\n\n")

========== init output: ========
>>>


========== read_pdf output: ========
>>>


Extracted Information: **Sector:**
- **Detailed Summary:**
  The RFP is issued by the Housing Opportunities Commission (HOC) of Montgomery County, Maryland, seeking comprehensive software development services for two potential projects: HOC Freedom of Information Act (FOIA) Request System and HOC Meeting Tracking System. The purpose is to develop business metrics tied to organizational processes to identify areas for improvement. The existing HOC application software includes various technologies and platforms such as Microsoft SQL Server, Windows 7 and 10, G-Suite, AWS, HTML, Java, JavaScript, Visual Basic, ASP, .NET, C#, AODocs, and Voyager by Yardi.
- **References from RFP text:**
  - "HOC seeks comprehensive software development services for 2 potential projects. The potential projects are HOC Freedom of Information Act (FOIA) Request and HOC Meeting Tracking Systems."
  - "HOC also plans to develop b